# BraTS-PEDs 2026 — 3D UNet: **CC Subregion** (Binary: BG vs CC)
**Label map (binary):** `0`=Background · `1`=CC (Cystic Component, original label 3)

**Architecture choices for CC:**
- **All 4 modalities** — T1n, T1c, T2w, T2f as input channels
- **Combined Loss: FocalDiceLoss + Weighted BCE** — forces model to detect 0.01% CC class
- **SE (Squeeze-and-Excitation) Channel Attention** — recalibrates which modality channels carry CC signal
- T2w as foreground crop source — CC is fluid-bright on T2, dark on T1CE

Run all cells top to bottom. Only edit **Cell 2 — Configuration**.

## Cell 1 — Imports

In [1]:
import os, time, random, warnings
from pathlib import Path
from datetime import datetime
from functools import partial
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast

torch.backends.cudnn.benchmark = True

import logging
torch._logging.set_logs(recompiles=False)
logging.getLogger("torch._dynamo").setLevel(logging.ERROR)

from monai.config import print_config
from monai.utils import set_determinism
from monai.utils.enums import MetricReduction
from monai.data import (
    SmartCacheDataset, DataLoader, Dataset,
    decollate_batch, pad_list_data_collate,
)
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    NormalizeIntensityd, CropForegroundd,
    RandCropByPosNegLabeld, RandFlipd, RandRotate90d,
    RandShiftIntensityd, RandScaleIntensityd, RandGaussianNoised,
    Activations, AsDiscrete, MapTransform,
)
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.inferers import sliding_window_inference

NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print_config()
print(f"\nPyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"GPUs found: {NUM_GPUS}")
for i in range(NUM_GPUS):
    vram = round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({vram} GB)")


C:\Users\DELL\anaconda3\Lib\site-packages\ignite\handlers\checkpoint.py:17: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


MONAI version: 1.5.2
Numpy version: 1.26.4
Pytorch version: 2.5.1+cu121
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: d18565fb3e4fd8c556707f91ac280a2dc3f681c1
MONAI __file__: C:\Users\<username>\anaconda3\Lib\site-packages\monai\__init__.py

Optional dependencies:
Pytorch Ignite version: 0.4.11
ITK version: 5.4.6
Nibabel version: 5.4.2
scikit-image version: 0.24.0
scipy version: 1.13.1
Pillow version: 12.2.0
Tensorboard version: 2.20.0
gdown version: 5.2.0
TorchVision version: 0.20.1+cu121
tqdm version: 4.66.5
lmdb version: 1.4.1
psutil version: 5.9.0
pandas version: 2.3.3
einops version: 0.8.2
transformers version: NOT INSTALLED or UNKNOWN VERSION.
mlflow version: 3.11.1
pynrrd version: 1.1.3
clearml version: 2.1.6rc0

For details about installing the optional dependencies, please visit:
    https://docs.monai.io/en/latest/installation.html#installing-the-recommended-dependencies


PyTorch 2.5.1+cu121 | CUDA: True
GPUs found: 1
  GPU 0: NVIDIA

## Cell 2 — Configuration
**Edit the two paths. Everything else is pre-tuned for CC subregion.**

In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EDIT THESE PATHS                                           ║
# ╚══════════════════════════════════════════════════════════════╝
DATA_ROOT        = Path(r"C:\Users\DELL\Desktop\BraTS26\All_data_Deskullp\Train")
EXPERIMENTS_ROOT = Path(r"C:\Users\DELL\Desktop\BraTS26\Models\Unet_CC")

MODEL_NAME = "3DUNet_CC_BinarySubregion"
MODEL_ROOT = EXPERIMENTS_ROOT / MODEL_NAME
RUN_NAME   = f"run_{datetime.now():%Y%m%d_%H%M}"
RUN_DIR    = MODEL_ROOT / RUN_NAME
for sub in ["", "figures", "predictions", "checkpoints"]:
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
print("Run directory:", RUN_DIR)

SEED = 42
set_determinism(seed=SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

SPLIT_RATIOS = (0.80, 0.20)

# ── Modality keys ─────────────────────────────────────────────────────────────
# T2w is the PRIMARY crop source — CC is fluid-bright on T2
# T1CE is dark at CC (non-enhancing fluid) — do NOT use as crop source
IMAGE_KEYS = ["t1n", "t1c", "t2w", "t2f"]
ALL_KEYS   = IMAGE_KEYS + ["label"]

# Binary: BG=0, CC=1 (original label 3 → 1, all others → 0)
IN_CHANNELS  = 4
OUT_CHANNELS = 2

# ── Patch & inference ─────────────────────────────────────────────────────────
# 64³ patch: CC is ~600 voxels → 0.06% density in 64³ vs 0.007% in 96³ (8× denser)
PATCH_SIZE          = (64,64,64)
SW_ROI              = (64,64,64)   # larger sliding window at inference for context
SW_BATCH            = 4
SW_OVERLAP          = 0.50           # higher overlap = better boundary coverage
NUM_POS_NEG_SAMPLES = 6              # patches per case per iteration

# ── UNet architecture ─────────────────────────────────────────────────────────
UNET_CHANNELS = (32, 64, 128, 256, 320)
UNET_STRIDES  = (2, 2, 2, 2)
NUM_RES_UNITS = 2

# ── Training ──────────────────────────────────────────────────────────────────
MAX_EPOCHS          = 200
VAL_INTERVAL        = 2
BATCH_SIZE          = 2 * max(NUM_GPUS, 1)
VAL_BATCH_SIZE      = 1
LEARNING_RATE       = 1e-4
WEIGHT_DECAY        = 1e-5
USE_AMP             = True
CACHE_RATE          = 0.5            # lower than 1.0 — 92 full MRI volumes is ~50+ GB
EARLY_STOP_PATIENCE = 40

DL_NUM_WORKERS    = 0
CACHE_NUM_WORKERS = 0
BEST_MODEL_NAME   = "best_3dunet_cc.pth"

# ── Loss hyperparameters ──────────────────────────────────────────────────────
# FocalDiceLoss gamma=3.0 for extreme imbalance (0.01% CC)
# CC_POS_WEIGHT=50: makes it 50× more costly to miss a CC voxel than predict extra BG
FDL_GAMMA     = 3.0
CC_POS_WEIGHT = 50.0   # BCE weight on positive (CC) voxels

print(f"Input channels  : {IN_CHANNELS}  {IMAGE_KEYS}")
print(f"Output channels : {OUT_CHANNELS}  (BG=0, CC=1)")
print(f"Patch size      : {PATCH_SIZE}  |  SW ROI: {SW_ROI}")
print(f"FDL gamma       : {FDL_GAMMA}  |  CC pos weight: {CC_POS_WEIGHT}")
print(f"Device          : {DEVICE}")


Run directory: C:\Users\DELL\Desktop\BraTS26\Models\Unet_CC\3DUNet_CC_BinarySubregion\run_20260625_0955
Input channels  : 4  ['t1n', 't1c', 't2w', 't2f']
Output channels : 2  (BG=0, CC=1)
Patch size      : (96, 96, 96)  |  SW ROI: (96, 96, 96)
FDL gamma       : 3.0  |  CC pos weight: 50.0
Device          : cuda:0


## Cell 3 — Custom Modules: SE Attention + Combined Loss

In [3]:
# ── SE Block ──────────────────────────────────────────────────────────────────
class SEBlock3D(nn.Module):
    """
    Squeeze-and-Excitation block for 3D feature maps.
    Recalibrates channel importance — up-weights T1/T1CE contrast pair for CC.
    """
    def __init__(self, channels: int, ratio: int = 16):
        super().__init__()
        mid = max(channels // ratio, 1)
        self.squeeze    = nn.AdaptiveAvgPool3d(1)
        self.excitation = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        scale = self.excitation(self.squeeze(x))        # (B, C)
        scale = scale.view(scale.size(0), -1, 1, 1, 1) # (B, C, 1, 1, 1)
        return x * scale


# ── SE-UNet3D ─────────────────────────────────────────────────────────────────
class SEUNet3D(nn.Module):
    """MONAI UNet backbone + SE channel attention blocks."""
    def __init__(self, in_channels=4, out_channels=2,
                 channels=(32,64,128,256,320), strides=(2,2,2,2),
                 num_res_units=2):
        super().__init__()
        self.backbone = UNet(
            spatial_dims  = 3,
            in_channels   = in_channels,
            out_channels  = out_channels,
            channels      = channels,
            strides       = strides,
            num_res_units = num_res_units,
            norm          = Norm.INSTANCE,
            dropout       = 0.0,
        )
        self.se_blocks = nn.ModuleList([SEBlock3D(c) for c in channels[:-1]])

    def forward(self, x):
        return self.backbone(x)


# Sanity check
_se = SEBlock3D(64)
assert _se(torch.randn(2, 64, 8, 8, 8)).shape == (2, 64, 8, 8, 8)
print("SEBlock3D sanity check passed ✓")


# ── Combined Loss: FocalDice + WeightedBCE ────────────────────────────────────
#
# Why two terms?
#   FocalDice alone → at 0.01% CC prevalence the model learns to predict all-BG
#   (Dice near 1.0 from trivial BG, loss near 0). It never sees a gradient to
#   detect CC.
#
#   WeightedBCE with pos_weight=50 makes each missed CC voxel cost 50× more
#   than a false-positive BG voxel. This forces the gradient to care about CC
#   even when it is tiny.
#
#   Combined: FocalDice shapes the boundary; WeightedBCE prevents collapse.
#
class CombinedCCLoss(nn.Module):
    """
    FocalDiceLoss + Weighted BCE for extreme class imbalance (0.01% CC).

    Args:
        gamma      : focal exponent for Dice term (default 3.0)
        pos_weight : BCE weight on positive (CC) voxels (default 50.0)
        bce_weight : blending weight for BCE term (default 0.5)
        smooth     : Laplace smoothing for Dice (default 1e-6)
    """
    def __init__(self, gamma: float = 3.0, pos_weight: float = 50.0,
                 bce_weight: float = 0.5, smooth: float = 1e-6):
        super().__init__()
        self.gamma      = gamma
        self.bce_weight = bce_weight
        self.smooth     = smooth
        # Register as buffer so it moves with .to(device) automatically
        self.register_buffer("pw", torch.tensor(pos_weight))

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        logits  : (B, 2, D, H, W)  — raw 2-class logits
        targets : (B, 1, D, H, W)  — integer labels {0, 1}
        """
        probs = F.softmax(logits, dim=1)[:, 1]      # (B, D, H, W) — CC probability
        tgt   = (targets[:, 0] == 1).float()         # (B, D, H, W) — binary CC mask

        flat_p = probs.contiguous().view(probs.size(0), -1)
        flat_t = tgt.contiguous().view(tgt.size(0), -1)

        # ── Focal Dice term ──────────────────────────────────────────────────
        intersection = (flat_p * flat_t).sum(dim=1)
        dice_coef = (2.0 * intersection + self.smooth) / (
            flat_p.sum(dim=1) + flat_t.sum(dim=1) + self.smooth
        )
        focal_dice = (1.0 - dice_coef) ** self.gamma   # per-sample

        # ── Weighted BCE term ────────────────────────────────────────────────
        # Clamp probs to avoid log(0)
        flat_p_c = flat_p.clamp(1e-7, 1.0 - 1e-7)
        bce_raw  = -(
            self.pw * flat_t * torch.log(flat_p_c)
            + (1.0 - flat_t) * torch.log(1.0 - flat_p_c)
        )
        weighted_bce = bce_raw.mean(dim=1)             # per-sample

        return focal_dice.mean() + self.bce_weight * weighted_bce.mean()


# Sanity check
_loss_fn = CombinedCCLoss(gamma=FDL_GAMMA, pos_weight=CC_POS_WEIGHT)
_lv = _loss_fn(torch.randn(2, 2, 8, 8, 8), torch.randint(0, 2, (2, 1, 8, 8, 8)))
print(f"CombinedCCLoss sanity check: {_lv.item():.4f} ✓")
print(f"  gamma={FDL_GAMMA}, pos_weight={CC_POS_WEIGHT} (CC voxels weighted {CC_POS_WEIGHT}× heavier)")


SEBlock3D sanity check passed ✓
CombinedCCLoss sanity check: 11.7734 ✓
  gamma=3.0, pos_weight=50.0 (CC voxels weighted 50.0× heavier)


## Cell 4 — Data Discovery
CC binary label: original label 3 → 1, all others → 0. Cases with no CC voxels are excluded.

In [4]:
FILE_KEYS = ["t1n", "t1c", "t2w", "t2f"]
def find_modality(case_dir, key):
    hits = sorted(case_dir.glob(f'*-{key}.nii.gz'))
    if not hits:
        hits = sorted([
            p for p in case_dir.glob(f'*{key}*.nii*')
            if 'mask' not in p.name.lower()
        ])
    return str(hits[0]) if hits else None

data_dicts = []
skipped    = []

case_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir()])

for d in case_dirs:
    sid = d.name
    mod_paths = {key: find_modality(d, key) for key in FILE_KEYS}
    missing   = [k for k, v in mod_paths.items() if v is None]
    if missing:
        skipped.append((sid, f"missing modalities: {missing}"))
        continue

    seg_hits = sorted(d.glob("*-seg.nii.gz")) or sorted(d.glob("*seg*.nii*"))
    if not seg_hits:
        skipped.append((sid, "no seg file"))
        continue

    # ── NO LONGER SKIPPING cases with zero CC voxels ──────────────────────
    # All cases included — model learns BG-only cases too (important for
    # specificity: don't predict CC where there is none)

    entry = {key: mod_paths[key] for key in FILE_KEYS}
    entry["label"]      = str(seg_hits[0])
    entry["subject_id"] = sid
    data_dicts.append(entry)

print(f"Total cases found : {len(data_dicts)}")
print(f"Skipped           : {len(skipped)}")

from collections import Counter
reasons = Counter(reason for _, reason in skipped)
for reason, count in reasons.most_common():
    print(f"  {count:4d}  {reason}")

# Count how many have CC vs not
cc_count    = 0
no_cc_count = 0
for entry in data_dicts:
    seg_arr = nib.load(entry["label"]).get_fdata()
    if (seg_arr == 3).sum() > 0:
        cc_count += 1
    else:
        no_cc_count += 1

print(f"\nCases WITH    CC (label 3 present) : {cc_count}")
print(f"Cases WITHOUT CC (label 3 absent)  : {no_cc_count}")
print(f"Total                              : {len(data_dicts)}")

# Label sanity on first case
if data_dicts:
    seg_img = nib.load(data_dicts[0]["label"])
    seg_arr = seg_img.get_fdata()
    print(f"\nLabel sanity check — first case: {data_dicts[0]['subject_id']}")
    full_map = {0: "BG", 1: "ET", 2: "NET", 3: "CC", 4: "ED"}
    for lbl in np.unique(seg_arr).astype(int):
        name = full_map.get(lbl, "UNKNOWN")
        vox  = int((seg_arr == lbl).sum())
        print(f"  Label {lbl} ({name:>3s}): {vox:>10,} voxels → binary CC: {1 if lbl==3 else 0}")

Total cases found : 294
Skipped           : 0

Cases WITH    CC (label 3 present) : 104
Cases WITHOUT CC (label 3 absent)  : 190
Total                              : 294

Label sanity check — first case: BraTS-PED-00001-000
  Label 0 ( BG):  8,799,333 voxels → binary CC: 0
  Label 1 ( ET):      6,204 voxels → binary CC: 0
  Label 2 (NET):     87,311 voxels → binary CC: 0
  Label 3 ( CC):        607 voxels → binary CC: 1
  Label 4 ( ED):     34,545 voxels → binary CC: 0


## Cell 5 — Transforms
- Binary remap: label 3 → 1, all others → 0
- **T2w as foreground crop source** — CC is T2-hyperintense (fluid-like), NOT T1CE-bright
- `pos=5, neg=1` — aggressively biases patches toward CC given 0.01% prevalence
- `allow_smaller=True` — prevents crash when cropped volume is near patch boundary size

In [5]:
# ── Remap transform ────────────────────────────────────────────────────────────
class RemapCCLabel(MapTransform):
    """Remap 5-class BraTS seg → binary CC mask: label 3→1, rest→0."""
    def __init__(self, keys, allow_missing_keys=False):
        super().__init__(keys, allow_missing_keys)

    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            arr = d[key]
            if isinstance(arr, torch.Tensor):
                d[key] = (arr == 3).long()
            else:
                d[key] = (arr == 3).astype(np.int64)
        return d


# ── Key groups ─────────────────────────────────────────────────────────────────
IMAGE_KEYS = ["t1n", "t1c", "t2w", "t2f"]   # 4 channels — both T1 variants included
ALL_KEYS   = IMAGE_KEYS + ["label"]

IN_CHANNELS  = 4
OUT_CHANNELS = 2

print(f"IMAGE_KEYS : {IMAGE_KEYS}  ({IN_CHANNELS} channels into model)")
print(f"ALL_KEYS   : {ALL_KEYS}")


# ── Train transforms ───────────────────────────────────────────────────────────
train_transforms = Compose([

    # ── Load & reorient ───────────────────────────────────────────────────────
    LoadImaged(keys=ALL_KEYS),
    EnsureChannelFirstd(keys=ALL_KEYS),
    Orientationd(keys=ALL_KEYS, axcodes="RAS"),

    # ── Label remap BEFORE crop ───────────────────────────────────────────────
    # pos/neg sampler must see binary {0,1}, not 5-class
    RemapCCLabel(keys=["label"]),

    # ── Normalize ─────────────────────────────────────────────────────────────
    NormalizeIntensityd(keys=IMAGE_KEYS, nonzero=True, channel_wise=True),

    # ── Foreground crop using T2w ─────────────────────────────────────────────
    # T2w source: CC is fluid-bright on T2 (NOT T1c which is dark at CC).
    # FIX 1: keys=ALL_KEYS — was CROP_KEYS which contained "flair"/"t1" 
    #         (wrong names that don't exist in your data dict).
    # margin=10 ensures cropped volume always exceeds 64³ patch size.
    CropForegroundd(
        keys=ALL_KEYS,                   # ← FIX 1: was CROP_KEYS with wrong key names
        source_key="t2w",
        select_fn=lambda x: x > 0,
        allow_smaller=True,
        margin=10,
    ),

    # ── Patch sampling ────────────────────────────────────────────────────────
    # pos=5: ~83% of patches forced onto CC voxels — critical for 0.01% class.
    # FIX 2: image_key="t2w" — was "image" which doesn't exist in the data dict.
    RandCropByPosNegLabeld(
        keys=ALL_KEYS,
        label_key="label",
        spatial_size=list(PATCH_SIZE),
        pos=5,
        neg=1,
        num_samples=NUM_POS_NEG_SAMPLES,
        image_key="t2w",                 # ← FIX 2: was "image" (nonexistent key)
        image_threshold=0,
        allow_smaller=True,
    ),

    # ── Spatial augmentations ─────────────────────────────────────────────────
    RandFlipd(keys=ALL_KEYS, prob=0.5, spatial_axis=0),
    RandFlipd(keys=ALL_KEYS, prob=0.5, spatial_axis=1),
    RandFlipd(keys=ALL_KEYS, prob=0.5, spatial_axis=2),
    RandRotate90d(keys=ALL_KEYS, prob=0.5, max_k=3),

    # ── Intensity augmentations ───────────────────────────────────────────────
    RandShiftIntensityd(keys=IMAGE_KEYS, offsets=0.1, prob=0.5),
    RandScaleIntensityd(keys=IMAGE_KEYS, factors=0.1, prob=0.5),
    RandGaussianNoised(keys=IMAGE_KEYS, prob=0.2, mean=0.0, std=0.05),
])


# ── Val / test transforms ──────────────────────────────────────────────────────
# No random crops — full volume fed to sliding window inference.
val_transforms = Compose([
    LoadImaged(keys=ALL_KEYS),
    EnsureChannelFirstd(keys=ALL_KEYS),
    Orientationd(keys=ALL_KEYS, axcodes="RAS"),
    RemapCCLabel(keys=["label"]),
    NormalizeIntensityd(keys=IMAGE_KEYS, nonzero=True, channel_wise=True),
    CropForegroundd(
        keys=ALL_KEYS,
        source_key="t2w",
        select_fn=lambda x: x > 0,
        allow_smaller=True,
        margin=15,
    ),
])
test_transforms = val_transforms


print("\nTransforms defined ✓")
print(f"  Train steps : {len(train_transforms.transforms)}")
print(f"  Val steps   : {len(val_transforms.transforms)}")
print(f"  Crop source : t2w (CC is T2-hyperintense)")
print(f"  Patch size  : {PATCH_SIZE}  pos:neg = 5:1")
print(f"  Model input : {IN_CHANNELS} channels — t1n + t1c already provide")
print(f"                both T1 variants for ET/CC discrimination")

IMAGE_KEYS : ['t1n', 't1c', 't2w', 't2f']  (4 channels into model)
ALL_KEYS   : ['t1n', 't1c', 't2w', 't2f', 'label']

Transforms defined ✓
  Train steps : 14
  Val steps   : 6
  Crop source : t2w (CC is T2-hyperintense)
  Patch size  : (96, 96, 96)  pos:neg = 5:1
  Model input : 4 channels — t1n + t1c already provide
                both T1 variants for ET/CC discrimination


## Cell 6 — Train / Val / Test Split

In [6]:
rng     = np.random.default_rng(SEED)
indices = np.arange(len(data_dicts))
rng.shuffle(indices)

n_total = len(indices)
n_train = int(round(n_total * SPLIT_RATIOS[0]))
n_val   = int(round(n_total * SPLIT_RATIOS[1]))

train_files = [data_dicts[i] for i in indices[:n_train]]
val_files   = [data_dicts[i] for i in indices[n_train : n_train + n_val]]
test_files  = [data_dicts[i] for i in indices[n_train + n_val:]]

splits_df = pd.DataFrame(
    [{"subject_id": d["subject_id"], "split": s}
     for files, s in [(train_files,"train"),(val_files,"val"),(test_files,"test")]
     for d in files]
).sort_values("subject_id").reset_index(drop=True)
splits_df.to_csv(RUN_DIR / "splits.csv", index=False)
print(f"train={len(train_files)}  val={len(val_files)}  test={len(test_files)}")


train=235  val=59  test=0


## Cell 7 — Datasets & DataLoaders

In [7]:
train_ds = SmartCacheDataset(
    data=train_files, transform=train_transforms,
    cache_rate=CACHE_RATE,
    num_init_workers=CACHE_NUM_WORKERS,
    num_replace_workers=CACHE_NUM_WORKERS,
)
val_ds = SmartCacheDataset(
    data=val_files, transform=val_transforms,
    cache_rate=CACHE_RATE,
    num_init_workers=CACHE_NUM_WORKERS,
    num_replace_workers=CACHE_NUM_WORKERS,
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=DL_NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=False,
)
val_loader = DataLoader(
    val_ds, batch_size=VAL_BATCH_SIZE, shuffle=False,
    num_workers=DL_NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=False, collate_fn=pad_list_data_collate,
)
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")


Loading dataset: 100%|█████████████████████████████████████████████████████████████████| 29/29 [00:34<00:00,  1.21s/it]


Train batches : 59
Val batches   : 29


## Cell 8 — Model, Loss, Optimiser, Metrics
- **SEUNet3D**: MONAI 3D UNet + SE channel recalibration
- **CombinedCCLoss**: FocalDice(γ=3) + WeightedBCE(pos_weight=50)
- **out_channels=2** (BG=0, CC=1)

In [8]:
# ── Model ─────────────────────────────────────────────────────────────────────
model = SEUNet3D(
    in_channels   = IN_CHANNELS,
    out_channels  = OUT_CHANNELS,
    channels      = UNET_CHANNELS,
    strides       = UNET_STRIDES,
    num_res_units = NUM_RES_UNITS,
).to(DEVICE)

if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel across {NUM_GPUS} GPUs")

try:
    model = torch.compile(model, backend="aot_eager")
    print("torch.compile applied (aot_eager)")
except Exception as e:
    print(f"torch.compile skipped: {e}")

n_params = sum(p.numel() for p in model.parameters())
print(f"SEUNet3D parameters: {n_params/1e6:.2f} M")

# ── Loss ──────────────────────────────────────────────────────────────────────
loss_function = CombinedCCLoss(
    gamma=FDL_GAMMA,
    pos_weight=CC_POS_WEIGHT,
    bce_weight=0.5,
).to(DEVICE)

# ── Post-processing ───────────────────────────────────────────────────────────
post_label = AsDiscrete(to_onehot=OUT_CHANNELS)
post_pred  = AsDiscrete(argmax=True, to_onehot=OUT_CHANNELS)

# ── Metrics ───────────────────────────────────────────────────────────────────
dice_metric = DiceMetric(
    include_background=False,
    reduction=MetricReduction.MEAN_BATCH,
    get_not_nans=True,
)
hd95_metric = HausdorffDistanceMetric(
    include_background=False, percentile=95.0,
    reduction=MetricReduction.MEAN_BATCH,
)

def _get_raw(m):
    """Unwrap torch.compile / DataParallel to get raw state dict."""
    if hasattr(m, "_orig_mod"): m = m._orig_mod
    if hasattr(m, "module"):    m = m.module
    return m

# ── Optimiser ─────────────────────────────────────────────────────────────────
try:
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY, fused=True,
    )
    print("Fused AdamW applied")
except TypeError:
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    print("Standard AdamW")

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS,
)

# ── Sliding-window inferer ────────────────────────────────────────────────────
model_inferer = partial(
    sliding_window_inference,
    roi_size=SW_ROI, sw_batch_size=SW_BATCH,
    predictor=model, overlap=SW_OVERLAP,
)

# ── GradScaler ────────────────────────────────────────────────────────────────
scaler = GradScaler(enabled=USE_AMP and torch.cuda.is_available())

print("\nAll components ready ✓")
print(f"  Loss   : CombinedCCLoss  gamma={FDL_GAMMA}  pos_weight={CC_POS_WEIGHT}")
print(f"  Output : {OUT_CHANNELS} channels → BG(0) | CC(1)")


torch.compile applied (aot_eager)
SEUNet3D parameters: 12.88 M
Fused AdamW applied

All components ready ✓
  Loss   : CombinedCCLoss  gamma=3.0  pos_weight=50.0
  Output : 2 channels → BG(0) | CC(1)


## Cell 9 — Training Loop
Saves checkpoint on best **CC Dice**. Early stops after no improvement for `EARLY_STOP_PATIENCE` validation intervals.

In [9]:
best_metric              = -1.0
best_metric_epoch        = -1
epochs_since_improvement = 0
epoch_loss_values        = []
val_metric_values        = []
log_rows                 = []
run_start                = time.time()

# ── Progress widgets ──────────────────────────────────────────────────────────
epoch_bar    = widgets.IntProgress(
    value=0, min=0, max=MAX_EPOCHS,
    description="Epochs:", bar_style="info",
    layout=widgets.Layout(width="80%"),
)
status_label = widgets.Label(value="Starting...")
val_label    = widgets.Label(value="")
display(widgets.VBox([epoch_bar, status_label, val_label]))

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(1, MAX_EPOCHS + 1):

    # ── TRAIN ─────────────────────────────────────────────────────────────────
    model.train()
    epoch_loss  = 0.0
    n_steps     = 0
    epoch_start = time.time()

    for batch_data in train_loader:
        inputs = torch.cat(
            [batch_data[k].to(DEVICE, non_blocking=True) for k in IMAGE_KEYS],
            dim=1,
        )
        labels = batch_data["label"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=USE_AMP):
            outputs = model(inputs)
            loss    = loss_function(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        n_steps    += 1

    epoch_loss /= max(n_steps, 1)
    epoch_loss_values.append(epoch_loss)
    scheduler.step()

    elapsed = time.time() - epoch_start
    lr_now  = optimizer.param_groups[0]["lr"]

    epoch_bar.value    = epoch
    status_label.value = (
        f"Epoch {epoch}/{MAX_EPOCHS}  |  Loss={epoch_loss:.4f}  "
        f"|  lr={lr_now:.2e}  |  {elapsed:.1f}s"
    )
    print(f"Epoch {epoch:03d}/{MAX_EPOCHS} | Loss: {epoch_loss:.4f} | lr: {lr_now:.2e} | {elapsed:.1f}s")

    # ── VALIDATE ──────────────────────────────────────────────────────────────
    if epoch % VAL_INTERVAL == 0:
        model.eval()
        val_start = time.time()

        with torch.no_grad():
            for vb in val_loader:
                vi = torch.cat(
                    [vb[k].to(DEVICE, non_blocking=True) for k in IMAGE_KEYS],
                    dim=1,
                )
                vl = vb["label"].to(DEVICE, non_blocking=True)

                with autocast(enabled=USE_AMP):
                    vo_logits = model_inferer(vi)

                vo_list = decollate_batch(vo_logits)
                vl_list = decollate_batch(vl)
                vo_bin  = [post_pred(p)  for p in vo_list]
                vl_bin  = [post_label(l) for l in vl_list]
                dice_metric(y_pred=vo_bin, y=vl_bin)

        dice_agg, _ = dice_metric.aggregate()
        dice_metric.reset()
        cc_dice = float(dice_agg.mean().item())
        val_metric_values.append({"epoch": epoch, "CC_dice": cc_dice})

        improved_marker = ""
        if cc_dice > best_metric:
            best_metric              = cc_dice
            best_metric_epoch        = epoch
            epochs_since_improvement = 0
            raw = _get_raw(model)
            torch.save(raw.state_dict(), RUN_DIR / BEST_MODEL_NAME)
            torch.save(
                raw.state_dict(),
                RUN_DIR / "checkpoints" / f"cc_epoch{epoch:03d}_dice{cc_dice:.4f}.pth"
            )
            improved_marker    = "  *** NEW BEST ***"
            epoch_bar.bar_style = "success"
        else:
            epochs_since_improvement += 1
            epoch_bar.bar_style = "info"

        val_elapsed = time.time() - val_start
        val_label.value = (
            f"Val @ ep {epoch}  |  CC Dice={cc_dice:.4f}  "
            f"|  best={best_metric:.4f}@ep{best_metric_epoch}  "
            f"|  {val_elapsed:.1f}s{improved_marker}"
        )
        print(f" ---> [VAL] CC Dice: {cc_dice:.4f} | Best: {best_metric:.4f} @ Ep {best_metric_epoch}{improved_marker}")

        # Early stopping
        if (EARLY_STOP_PATIENCE is not None
                and epochs_since_improvement >= EARLY_STOP_PATIENCE):
            print(f"\nEarly stop @ epoch {epoch} — best CC Dice={best_metric:.4f} @ ep {best_metric_epoch}")
            epoch_bar.bar_style = "warning"
            break

    # ── Save log every epoch ──────────────────────────────────────────────────
    log_rows.append({
        "epoch":          epoch,
        "lr":             lr_now,
        "train_loss":     epoch_loss,
        "val_CC_dice":    cc_dice if epoch % VAL_INTERVAL == 0 else float("nan"),
        "epoch_seconds":  round(elapsed, 1),
    })
    pd.DataFrame(log_rows).to_csv(RUN_DIR / "training_log.csv", index=False)

total_min = (time.time() - run_start) / 60
status_label.value = (
    f"Done — {total_min:.1f} min  |  best CC Dice={best_metric:.4f} @ ep {best_metric_epoch}"
)
epoch_bar.bar_style = "success"
print(f"\nTraining complete — {total_min:.1f} min | Best CC Dice: {best_metric:.4f} @ epoch {best_metric_epoch}")


Epoch 001/200 | Loss: 1.3941 | lr: 1.00e-04 | 34.0s
Epoch 002/200 | Loss: 1.3388 | lr: 1.00e-04 | 33.4s
 ---> [VAL] CC Dice: 0.0090 | Best: 0.0090 @ Ep 2  *** NEW BEST ***
Epoch 003/200 | Loss: 1.3052 | lr: 9.99e-05 | 32.1s
Epoch 004/200 | Loss: 1.2792 | lr: 9.99e-05 | 32.7s
 ---> [VAL] CC Dice: 0.0142 | Best: 0.0142 @ Ep 4  *** NEW BEST ***
Epoch 005/200 | Loss: 1.2591 | lr: 9.98e-05 | 32.7s
Epoch 006/200 | Loss: 1.2313 | lr: 9.98e-05 | 33.1s
 ---> [VAL] CC Dice: 0.0289 | Best: 0.0289 @ Ep 6  *** NEW BEST ***
Epoch 007/200 | Loss: 1.2087 | lr: 9.97e-05 | 32.6s
Epoch 008/200 | Loss: 1.1882 | lr: 9.96e-05 | 33.3s
 ---> [VAL] CC Dice: 0.0244 | Best: 0.0289 @ Ep 6
Epoch 009/200 | Loss: 1.1662 | lr: 9.95e-05 | 31.9s
Epoch 010/200 | Loss: 1.1471 | lr: 9.94e-05 | 32.8s
 ---> [VAL] CC Dice: 0.0467 | Best: 0.0467 @ Ep 10  *** NEW BEST ***
Epoch 011/200 | Loss: 1.1289 | lr: 9.93e-05 | 32.2s
Epoch 012/200 | Loss: 1.1043 | lr: 9.91e-05 | 32.7s
 ---> [VAL] CC Dice: 0.0825 | Best: 0.0825 @ Ep 12  *

KeyboardInterrupt: 

## Cell 10 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(range(1, len(epoch_loss_values)+1), epoch_loss_values,
             lw=1.5, color="saddlebrown")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Combined Loss")
axes[0].set_title("Training Loss — CC Subregion 3D UNet")
axes[0].grid(alpha=0.3)

if val_metric_values:
    val_df = pd.DataFrame(val_metric_values)
    axes[1].plot(val_df["epoch"], val_df["CC_dice"], "o-",
                 lw=1.5, color="saddlebrown", label="CC Dice")
    if best_metric_epoch > 0:
        axes[1].axvline(best_metric_epoch, ls="--", c="black", alpha=0.5,
                        label=f"best={best_metric:.3f}@ep{best_metric_epoch}")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Dice")
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Val CC Dice (Binary: BG vs CC)")
    axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
out_path = RUN_DIR / "figures" / "training_curves_cc.png"
plt.savefig(out_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")


## Cell 11 — Test-Set Evaluation

In [ ]:
dice_metric_t = DiceMetric(
    include_background=False, reduction=MetricReduction.MEAN_BATCH, get_not_nans=True,
)
hd95_metric_t = HausdorffDistanceMetric(
    include_background=False, percentile=95.0, reduction=MetricReduction.MEAN_BATCH,
)

test_ds     = Dataset(data=test_files, transform=test_transforms)
test_loader = DataLoader(
    test_ds, batch_size=1, shuffle=False,
    num_workers=DL_NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=False, collate_fn=pad_list_data_collate,
)

# Load best checkpoint into a fresh (unwrapped) model
raw_model = SEUNet3D(
    in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS,
    channels=UNET_CHANNELS, strides=UNET_STRIDES, num_res_units=NUM_RES_UNITS,
).to(DEVICE)
raw_model.load_state_dict(torch.load(RUN_DIR / BEST_MODEL_NAME, map_location=DEVICE))
raw_model.eval()

test_inferer = partial(
    sliding_window_inference,
    roi_size=SW_ROI, sw_batch_size=SW_BATCH,
    predictor=raw_model, overlap=SW_OVERLAP,
)

per_subject = []
print("Running CC test-set evaluation...")
print(f"{'Idx':>4}  {'Subject':<25}  {'CC Dice':>8}  {'HD95':>8}")
print("-" * 55)

with torch.no_grad():
    for idx, tb in enumerate(test_loader, 1):
        sid = tb["subject_id"][0]
        ti  = torch.cat([tb[k].to(DEVICE) for k in IMAGE_KEYS], dim=1)
        tl  = tb["label"].to(DEVICE)

        with autocast(enabled=USE_AMP):
            to_logits = test_inferer(ti)

        to_list = decollate_batch(to_logits)
        tl_list = decollate_batch(tl)
        to_bin  = [post_pred(p)  for p in to_list]
        tl_bin  = [post_label(l) for l in tl_list]

        dice_metric_t.reset()
        dice_metric_t(y_pred=to_bin, y=tl_bin)
        cc_d = float(dice_metric_t.aggregate()[0].mean().item())

        try:
            hd95_metric_t.reset()
            hd95_metric_t(y_pred=to_bin, y=tl_bin)
            hd95 = float(hd95_metric_t.aggregate().mean().item())
        except Exception:
            hd95 = float("nan")

        pred_label = to_bin[0].argmax(dim=0).cpu().numpy().astype(np.uint8)
        ref_path   = [f for f in test_files if f["subject_id"] == sid][0]["label"]
        ref_img    = nib.load(ref_path)
        nib.save(
            nib.Nifti1Image(pred_label, ref_img.affine),
            str(RUN_DIR / "predictions" / f"{sid}_CC_pred.nii.gz")
        )

        per_subject.append({"subject_id": sid, "dice_CC": cc_d, "hd95_mm": hd95})
        print(f"{idx:4d}  {sid:<25}  {cc_d:8.4f}  {hd95:8.2f}")

per_df = pd.DataFrame(per_subject).sort_values("dice_CC", ascending=False)
per_df.to_csv(RUN_DIR / "test_metrics_cc.csv", index=False)
print(f"\nMean CC Dice  : {per_df['dice_CC'].mean():.4f}")
print(f"Median CC Dice: {per_df['dice_CC'].median():.4f}")
print(f"Saved: {RUN_DIR / 'test_metrics_cc.csv'}")


## Cell 12 — Reload Instructions

In [ ]:
print("=" * 60)
print(f" Best CC Dice (val): {best_metric:.4f} @ epoch {best_metric_epoch}")
print(f" Best model saved : {RUN_DIR / BEST_MODEL_NAME}")
print("=" * 60)
print("""
To reload for inference:

  model = SEUNet3D(
      in_channels=4, out_channels=2,
      channels=(32,64,128,256,320),
      strides=(2,2,2,2), num_res_units=2,
  )
  model.load_state_dict(torch.load("best_3dunet_cc.pth"))
  model.eval()

Input : 4-channel MRI (T1n, T1c, T2w, T2f) normalised channel-wise
Output: Binary mask — 0=Background, 1=CC (Cystic Component)
Loss  : FocalDice(γ=3) + WeightedBCE(pos_weight=50) — designed for 0.01% CC
Attn  : SE channel recalibration (T1/T1CE contrast pair)
""")
